# E-Commerce Marketplace Analytics — SQLite Database Setup

## Purpose of SQLite in This Project
This notebook builds the single relational database that every later SQL and
Python analysis notebook will query against. SQLite stores the entire database
as one portable file (`database/ecommerce.db`), requiring no separate server
process, no installation, and no network configuration.

## Why SQLite Was Selected
- **Zero setup:** no database server to install, configure, or run in the
  background — the database is just a file.
- **Full SQL support** for everything this project needs: joins, subqueries,
  CTEs, and window functions are all supported natively.
- **Portable and reproducible:** the `.db` file (or the scripts that build
  it) can be committed to Git or shared directly, so anyone can reproduce
  the exact same database.
- **Appropriate scale:** at roughly 100,000 orders, this dataset comfortably
  fits SQLite's design envelope — a heavier server-based database would add
  operational complexity with no analytical benefit here.

## Advantages for a Portfolio Project
A reviewer can clone the repository, run one notebook, and immediately have
a working database to query — no server credentials, no Docker, no cloud
account. This lowers the barrier to someone actually trying the project
rather than just reading about it.

## Database Architecture
Six tables, matching the schema in `sql/00_create_schema.sql` and built from
the cleaned CSVs produced in `01_data_cleaning.ipynb`:

```
categories ──< products ──< order_items >── orders >── customers
                                              order_reviews >── orders
```

This notebook only builds and validates the database. No SQL analysis, KPI
calculation, EDA, or charting happens here — that begins in
`03_sql_analysis.ipynb`.


## Section 2 — Database Design

Each table exists for a specific reason:

| Table | Why it exists |
|---|---|
| `categories` | Central English/Portuguese category reference, so every product and every revenue-by-category KPI resolves to one consistent label per category. |
| `customers` | One row per real, deduplicated customer (`customer_unique_id`), enabling correct repeat-purchase and customer-value analysis. |
| `products` | Links each product to its category and basic physical attributes, needed to join `order_items` up to category-level analysis. |
| `orders` | The order lifecycle anchor — status, purchase date, delivery dates — that every time-based and delivery KPI depends on. |
| `order_items` | The core transactional fact table: one row per product per order, the source of all revenue calculations. |
| `order_reviews` | One row per order's final review score, enabling the delivery-performance vs. satisfaction analysis. |

The full column-level schema (data types, keys, nullability) is defined in
`sql/00_create_schema.sql`, executed in Section 3 below.


## Section 3 — Connect to SQLite and Execute the Schema

This section connects to (creating, if necessary) the database file, runs
`sql/00_create_schema.sql` in full, and immediately verifies that all six
tables were created.


In [1]:
from __future__ import annotations

import sqlite3
import warnings
from pathlib import Path
from typing import Dict, List

import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SQL_DIR = PROJECT_ROOT / "sql"
DATABASE_DIR = PROJECT_ROOT / "database"
REPORTS_DIR = PROJECT_ROOT / "reports"

DATABASE_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATABASE_DIR / "ecommerce.db"
SCHEMA_SQL_PATH = SQL_DIR / "00_create_schema.sql"
VALIDATION_SQL_PATH = SQL_DIR / "00_validation.sql"

# Table load order follows the schema's foreign key dependency chain:
# parents before children.
TABLE_LOAD_ORDER: List[str] = [
    "categories",
    "customers",
    "products",
    "orders",
    "order_items",
    "order_reviews",
]


In [2]:
def get_connection(db_path: Path) -> sqlite3.Connection:
    """Open a SQLite connection with foreign key enforcement turned on.

    SQLite disables foreign key constraint enforcement by default for
    backward compatibility, so PRAGMA foreign_keys = ON must be set on
    every new connection to actually enforce the FOREIGN KEY clauses
    declared in the schema.
    """
    connection = sqlite3.connect(db_path)
    connection.execute("PRAGMA foreign_keys = ON;")
    return connection


def build_schema(connection: sqlite3.Connection, schema_sql_path: Path) -> None:
    """Execute the full schema DDL script against the given connection."""
    if not schema_sql_path.exists():
        raise FileNotFoundError(f"Schema file not found: {schema_sql_path}")
    schema_sql = schema_sql_path.read_text(encoding="utf-8")
    connection.executescript(schema_sql)
    connection.commit()


# Fresh build: remove any existing database file so this notebook is
# always safely re-runnable from a clean state.
if DB_PATH.exists():
    DB_PATH.unlink()
    print(f"Removed existing database file at {DB_PATH} for a clean rebuild.")

conn = get_connection(DB_PATH)
build_schema(conn, SCHEMA_SQL_PATH)
print(f"Schema created successfully at {DB_PATH}")


Removed existing database file at /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics FINAL/database/ecommerce.db for a clean rebuild.
Schema created successfully at /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics FINAL/database/ecommerce.db


In [3]:
def list_tables(connection: sqlite3.Connection) -> List[str]:
    """Return all user-defined table names in the database."""
    cursor = connection.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
    )
    return [row[0] for row in cursor.fetchall()]


created_tables = list_tables(conn)
print("Tables created:", created_tables)

expected_tables = set(TABLE_LOAD_ORDER)
missing_tables = expected_tables - set(created_tables)
if missing_tables:
    raise RuntimeError(f"Schema creation incomplete — missing tables: {missing_tables}")
print("All expected tables are present.")


Tables created: ['categories', 'customers', 'order_items', 'order_reviews', 'orders', 'products']
All expected tables are present.


### Display Schema Information

`PRAGMA table_info()` is used to display the column-level definition of
every table exactly as SQLite understands it, confirming the schema matches
`sql/00_create_schema.sql`.


In [4]:
def show_table_info(connection: sqlite3.Connection, table_name: str) -> pd.DataFrame:
    """Return PRAGMA table_info() for a table as a readable DataFrame."""
    cursor = connection.execute(f"PRAGMA table_info({table_name});")
    columns = [desc[0] for desc in cursor.description]
    return pd.DataFrame(cursor.fetchall(), columns=columns)


for table in TABLE_LOAD_ORDER:
    print(f"--- {table} ---")
    display_df = show_table_info(conn, table)
    print(display_df.to_string(index=False))
    print()


--- categories ---
 cid                     name    type  notnull dflt_value  pk
   0              category_id INTEGER        0       None   1
   1 category_name_portuguese    TEXT        1       None   0
   2    category_name_english    TEXT        0       None   0

--- customers ---
 cid               name type  notnull dflt_value  pk
   0 customer_unique_id TEXT        0       None   1
   1      customer_city TEXT        0       None   0
   2     customer_state TEXT        0       None   0

--- products ---
 cid             name    type  notnull dflt_value  pk
   0       product_id    TEXT        0       None   1
   1      category_id INTEGER        0       None   0
   2 product_weight_g    REAL        0       None   0

--- orders ---
 cid                          name    type  notnull dflt_value  pk
   0                      order_id    TEXT        0       None   1
   1            customer_unique_id    TEXT        1       None   0
   2                  order_status    TEXT        1

## Section 5 — Load Clean Data

The corresponding CLI-equivalent commands are documented in
`sql/00_load_data.sql` for anyone loading the database outside of Python.
Here, the six cleaned CSVs produced in `01_data_cleaning.ipynb` are loaded
programmatically via pandas, in the same dependency order as the schema,
with row counts validated immediately after each load.


In [5]:
def load_csv_to_table(
    connection: sqlite3.Connection,
    csv_path: Path,
    table_name: str,
) -> int:
    """Load one cleaned CSV into its matching pre-created SQLite table."""
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Expected cleaned CSV not found: {csv_path}\n"
            "Run notebooks/01_data_cleaning.ipynb first."
        )

    df = pd.read_csv(csv_path)

    # Safety check: the cleaning notebook should already guarantee uniqueness.
    if table_name == "order_reviews":
        duplicate_review_ids = int(df["review_id"].duplicated().sum())
        duplicate_order_ids = int(df["order_id"].duplicated().sum())
        if duplicate_review_ids or duplicate_order_ids:
            raise ValueError(
                "order_reviews.csv still contains duplicates: "
                f"review_id={duplicate_review_ids}, order_id={duplicate_order_ids}. "
                "Restart and run 01_data_cleaning.ipynb, then rerun this notebook."
            )

    # Use append because the schema (including PK/FK constraints) was created above.
    df.to_sql(table_name, connection, if_exists="append", index=False)
    return len(df)


imported_row_counts: Dict[str, int] = {}

for table in TABLE_LOAD_ORDER:
    csv_path = DATA_PROCESSED_DIR / f"{table}.csv"
    n_rows = load_csv_to_table(conn, csv_path, table)
    imported_row_counts[table] = n_rows
    print(f"Loaded {table}: {n_rows:,} rows from {csv_path.name}")

conn.commit()


Loaded categories: 73 rows from categories.csv
Loaded customers: 96,096 rows from customers.csv
Loaded products: 32,951 rows from products.csv
Loaded orders: 99,441 rows from orders.csv
Loaded order_items: 112,650 rows from order_items.csv
Loaded order_reviews: 98,127 rows from order_reviews.csv


In [6]:
def get_table_row_count(connection: sqlite3.Connection, table_name: str) -> int:
    """Return the actual row count currently stored in a table."""
    cursor = connection.execute(f"SELECT COUNT(*) FROM {table_name};")
    return cursor.fetchone()[0]


import_validation = pd.DataFrame(
    [
        {
            "table": table,
            "rows_in_csv": imported_row_counts[table],
            "rows_in_database": get_table_row_count(conn, table),
        }
        for table in TABLE_LOAD_ORDER
    ]
)
import_validation["match"] = (
    import_validation["rows_in_csv"] == import_validation["rows_in_database"]
)
print(import_validation.to_string(index=False))

if not import_validation["match"].all():
    raise RuntimeError("Row count mismatch between a cleaned CSV and its loaded table — investigate before proceeding.")
print("\nAll table row counts match their source CSVs exactly.")


        table  rows_in_csv  rows_in_database  match
   categories           73                73   True
    customers        96096             96096   True
     products        32951             32951   True
       orders        99441             99441   True
  order_items       112650            112650   True
order_reviews        98127             98127   True

All table row counts match their source CSVs exactly.


## Section 6 — Integrity Validation

Every check defined in `sql/00_validation.sql` is executed here
programmatically, with results captured into readable summaries rather than
raw query output. A healthy database should return **zero rows** for every
check except the informational row-count and category-mapping queries,
which are expected to show real (non-error) values.


In [7]:
def run_query(connection: sqlite3.Connection, query: str) -> pd.DataFrame:
    """Run a single SQL query and return the result as a DataFrame."""
    return pd.read_sql_query(query, connection)


def check_primary_key_uniqueness_sql(connection: sqlite3.Connection, table: str, key_col: str) -> Dict[str, object]:
    """Confirm a primary key column has no duplicate values in the loaded table."""
    df = run_query(
        connection,
        f"SELECT {key_col}, COUNT(*) AS n FROM {table} GROUP BY {key_col} HAVING COUNT(*) > 1;",
    )
    return {"table": table, "key_column": key_col, "duplicate_key_rows": len(df), "passed": len(df) == 0}


pk_results = [
    check_primary_key_uniqueness_sql(conn, "customers", "customer_unique_id"),
    check_primary_key_uniqueness_sql(conn, "orders", "order_id"),
    check_primary_key_uniqueness_sql(conn, "products", "product_id"),
    check_primary_key_uniqueness_sql(conn, "categories", "category_id"),
    check_primary_key_uniqueness_sql(conn, "order_reviews", "review_id"),
]
pk_results_df = pd.DataFrame(pk_results)
pk_results_df


,table,key_column,duplicate_key_rows,passed
0,customers,customer_unique_id,0,True
1,orders,order_id,0,True
2,products,product_id,0,True
3,categories,category_id,0,True
4,order_reviews,review_id,0,True


In [8]:
# Composite key uniqueness for order_items (order_id, order_item_id)
composite_key_check = run_query(
    conn,
    """
    SELECT order_id, order_item_id, COUNT(*) AS n
    FROM order_items
    GROUP BY order_id, order_item_id
    HAVING COUNT(*) > 1;
    """,
)
print("order_items composite key violations:", len(composite_key_check))
composite_key_check


order_items composite key violations: 0


,order_id,order_item_id,n


In [9]:
def check_foreign_key_sql(
    connection: sqlite3.Connection,
    child_table: str,
    child_col: str,
    parent_table: str,
    parent_col: str,
    nullable_fk: bool = False,
) -> Dict[str, object]:
    """Confirm every non-null child value has a matching parent row."""
    null_filter = f"{child_table}.{child_col} IS NOT NULL AND" if nullable_fk else ""
    query = f"""
        SELECT {child_table}.{child_col}
        FROM {child_table}
        LEFT JOIN {parent_table} ON {child_table}.{child_col} = {parent_table}.{parent_col}
        WHERE {null_filter} {parent_table}.{parent_col} IS NULL;
    """
    df = run_query(connection, query)
    return {
        "relationship": f"{child_table}.{child_col} -> {parent_table}.{parent_col}",
        "orphan_rows": len(df),
        "passed": len(df) == 0,
    }


fk_results = [
    check_foreign_key_sql(conn, "orders", "customer_unique_id", "customers", "customer_unique_id"),
    check_foreign_key_sql(conn, "order_items", "order_id", "orders", "order_id"),
    check_foreign_key_sql(conn, "order_items", "product_id", "products", "product_id"),
    check_foreign_key_sql(conn, "products", "category_id", "categories", "category_id", nullable_fk=True),
    check_foreign_key_sql(conn, "order_reviews", "order_id", "orders", "order_id"),
]
fk_results_df = pd.DataFrame(fk_results)
fk_results_df


,relationship,orphan_rows,passed
0,orders.customer_unique_id -> customers.custome...,0,True
1,order_items.order_id -> orders.order_id,0,True
2,order_items.product_id -> products.product_id,0,True
3,products.category_id -> categories.category_id,0,True
4,order_reviews.order_id -> orders.order_id,0,True


In [10]:
# NOT NULL constraint spot-check across the columns defined as NOT NULL in the schema
not_null_checks = {
    "orders": ["customer_unique_id", "order_status", "order_estimated_delivery_date"],
    "order_items": ["order_id", "product_id", "price"],
    "order_reviews": ["order_id", "review_score", "review_creation_date"],
}

not_null_results = []
for table, columns in not_null_checks.items():
    for column in columns:
        df = run_query(conn, f"SELECT COUNT(*) AS n FROM {table} WHERE {column} IS NULL;")
        n_nulls = int(df.iloc[0]["n"])
        not_null_results.append({"table": table, "column": column, "unexpected_nulls": n_nulls, "passed": n_nulls == 0})

not_null_df = pd.DataFrame(not_null_results)
not_null_df


,table,column,unexpected_nulls,passed
0,orders,customer_unique_id,0,True
1,orders,order_status,0,True
2,orders,order_estimated_delivery_date,0,True
3,order_items,order_id,0,True
4,order_items,product_id,0,True
5,order_items,price,0,True
6,order_reviews,order_id,0,True
7,order_reviews,review_score,0,True
8,order_reviews,review_creation_date,0,True


In [11]:
# Review linkage: confirm exactly one review per order (post-cleaning expectation)
review_cardinality_check = run_query(
    conn,
    """
    SELECT order_id, COUNT(*) AS n_reviews
    FROM order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1;
    """,
)
print("Orders with more than one review:", len(review_cardinality_check))

# Customer linkage: every order must resolve to exactly one customer (already
# covered by the foreign key check above, re-confirmed here for completeness)
customer_linkage_check = run_query(
    conn,
    """
    SELECT o.order_id
    FROM orders o
    LEFT JOIN customers c ON o.customer_unique_id = c.customer_unique_id
    WHERE c.customer_unique_id IS NULL;
    """,
)
print("Orders with no matching customer:", len(customer_linkage_check))

# Category mapping integrity: informational count of products whose category
# has no English translation (expected to be small/zero, not an error state)
category_mapping_check = run_query(
    conn,
    """
    SELECT p.product_id, c.category_name_portuguese
    FROM products p
    JOIN categories c ON p.category_id = c.category_id
    WHERE c.category_name_english IS NULL;
    """,
)
print("Products with an untranslated category (informational, not an error):", len(category_mapping_check))
category_mapping_check


Orders with more than one review: 0
Orders with no matching customer: 0
Products with an untranslated category (informational, not an error): 13


,product_id,category_name_portuguese
0,0105b5323d24fc655f73052694dbbb3a,pc_gamer
1,6fd83eb3e0799b775e4f946bd66657c0,portateis_cozinha_e_preparadores_de_alimentos
2,5d923ead886c44b86845f69e50520c3e,portateis_cozinha_e_preparadores_de_alimentos
3,6727051471a0fc4a0e7737b57bff2549,pc_gamer
4,bed164d9d628cf0593003389c535c6e0,portateis_cozinha_e_preparadores_de_alimentos
5,1220978a08a6b29a202bc015b18250e9,portateis_cozinha_e_preparadores_de_alimentos
6,ae62bb0f95af63d64eae5f93dddea8d3,portateis_cozinha_e_preparadores_de_alimentos
7,1954739d84629e7323a4295812a3e0ec,portateis_cozinha_e_preparadores_de_alimentos
8,dbe520fb381ad695a7e1f2807d20c765,pc_gamer
9,c7a3f1a7f9eef146cc499368b578b884,portateis_cozinha_e_preparadores_de_alimentos


**Overall integrity verdict:** every check above should show `passed = True`
(zero orphan/duplicate/null rows), with the sole exception of the
category-mapping check, which is expected to surface the same small number
of untranslated categories already documented in
`reports/data_quality_report.md` — that is a known, accepted limitation,
not a database defect.


In [12]:
all_checks_passed = (
    pk_results_df["passed"].all()
    and fk_results_df["passed"].all()
    and not_null_df["passed"].all()
    and len(composite_key_check) == 0
    and len(review_cardinality_check) == 0
    and len(customer_linkage_check) == 0
)
print("All structural integrity checks passed:", all_checks_passed)


All structural integrity checks passed: True


## Section 7 — Database Inspection

A final, automated inspection pass: the full table list, `PRAGMA
table_info()` for every table, the declared foreign keys, current row
counts, and a small sample of records from each table.


In [13]:
print("Tables:", list_tables(conn))


Tables: ['categories', 'customers', 'order_items', 'order_reviews', 'orders', 'products']


In [14]:
def show_foreign_keys(connection: sqlite3.Connection, table_name: str) -> pd.DataFrame:
    """Return PRAGMA foreign_key_list() for a table as a readable DataFrame."""
    cursor = connection.execute(f"PRAGMA foreign_key_list({table_name});")
    columns = [desc[0] for desc in cursor.description]
    return pd.DataFrame(cursor.fetchall(), columns=columns)


for table in TABLE_LOAD_ORDER:
    fk_info = show_foreign_keys(conn, table)
    print(f"--- Foreign keys declared on {table} ---")
    print(fk_info.to_string(index=False) if not fk_info.empty else "(none)")
    print()


--- Foreign keys declared on categories ---
(none)

--- Foreign keys declared on customers ---
(none)

--- Foreign keys declared on products ---
 id  seq      table        from          to on_update on_delete match
  0    0 categories category_id category_id NO ACTION NO ACTION  NONE

--- Foreign keys declared on orders ---
 id  seq     table               from                 to on_update on_delete match
  0    0 customers customer_unique_id customer_unique_id NO ACTION NO ACTION  NONE

--- Foreign keys declared on order_items ---
 id  seq    table       from         to on_update on_delete match
  0    0 products product_id product_id NO ACTION NO ACTION  NONE
  1    0   orders   order_id   order_id NO ACTION NO ACTION  NONE

--- Foreign keys declared on order_reviews ---
 id  seq  table     from       to on_update on_delete match
  0    0 orders order_id order_id NO ACTION NO ACTION  NONE



In [15]:
final_row_counts = pd.DataFrame(
    [{"table": table, "row_count": get_table_row_count(conn, table)} for table in TABLE_LOAD_ORDER]
)
final_row_counts


,table,row_count
0,categories,73
1,customers,96096
2,products,32951
3,orders,99441
4,order_items,112650
5,order_reviews,98127


In [16]:
for table in TABLE_LOAD_ORDER:
    print(f"--- Sample records: {table} ---")
    sample = run_query(conn, f"SELECT * FROM {table} LIMIT 3;")
    print(sample.to_string(index=False))
    print()


--- Sample records: categories ---
 category_id category_name_portuguese category_name_english
           1             beleza_saude         health_beauty
           2   informatica_acessorios computers_accessories
           3               automotivo                  auto

--- Sample records: customers ---
              customer_unique_id         customer_city customer_state
861eff4711a542e4b93843c6dd7febb0                franca             SP
290c77bc529b7ac935b93aa66c333dc3 sao bernardo do campo             SP
060e732b5b29e8181a18229c7b0b2b5e             sao paulo             SP

--- Sample records: products ---
                      product_id  category_id  product_weight_g
1e9e8ef04dbcff4541ed26657ea517e5            7             225.0
3aa071139cb16b67ca9e5dea641aaa2f           47            1000.0
96bd76ec8810374ed1b65e291975717f            6             154.0

--- Sample records: orders ---
                        order_id               customer_unique_id order_status order_pur

## Section 8 — Validation Report

The full validation report is generated directly from this notebook's own
results (not hand-typed) and written to
`reports/database_validation_report.md`.


In [17]:
def build_database_validation_report(
    db_path: Path,
    tables: List[str],
    import_validation: pd.DataFrame,
    pk_results_df: pd.DataFrame,
    fk_results_df: pd.DataFrame,
    not_null_df: pd.DataFrame,
    composite_key_violations: int,
    review_cardinality_violations: int,
    category_mapping_check: pd.DataFrame,
    final_row_counts: pd.DataFrame,
) -> str:
    """Assemble the full database_validation_report.md content from live results."""
    lines: List[str] = []
    lines.append("# Database Validation Report — E-Commerce Marketplace Analytics\n")
    lines.append(
        f"Database file: `{db_path.relative_to(db_path.parents[1])}`. "
        "This report is generated programmatically from "
        "`02_sqlite_database_setup.ipynb`'s own validation results.\n"
    )

    lines.append("## Database Overview\n")
    lines.append(f"- Tables created: {', '.join(tables)}\n")

    lines.append("## Tables Created and Rows Imported\n")
    lines.append(import_validation.to_markdown(index=False))
    lines.append("\n")

    lines.append("## Final Row Counts (post-load)\n")
    lines.append(final_row_counts.to_markdown(index=False))
    lines.append("\n")

    lines.append("## Integrity Checks — Primary Keys\n")
    lines.append(pk_results_df.to_markdown(index=False))
    lines.append("\n")

    lines.append("## Integrity Checks — Foreign Keys\n")
    lines.append(fk_results_df.to_markdown(index=False))
    lines.append("\n")

    lines.append("## Integrity Checks — NOT NULL Constraints\n")
    lines.append(not_null_df.to_markdown(index=False))
    lines.append("\n")

    lines.append("## Constraint Validation Summary\n")
    lines.append(
        f"- Composite key violations in `order_items`: {composite_key_violations}\n"
        f"- Orders with more than one review: {review_cardinality_violations}\n"
        f"- Products with an untranslated category (informational, not an error): {len(category_mapping_check)}\n"
    )

    lines.append("## Known Limitations\n")
    lines.append(
        "- A small number of products map to a category with no English translation. "
        "This is inherited directly from the raw Olist data (documented in "
        "`reports/data_quality_report.md`) and is preserved here rather than papered over.\n"
        "- SQLite enforces foreign keys only when `PRAGMA foreign_keys = ON` is set per "
        "connection; any future script or tool connecting to this database file must set "
        "this pragma itself to get real-time enforcement.\n"
        "- No indexes beyond the primary/foreign key columns have been created yet — see "
        "Section 9 for planned (not yet implemented) index recommendations.\n"
    )

    lines.append("## Ready for SQL Analysis\n")
    lines.append(
        "All structural integrity checks passed (or, for the one informational "
        "category-mapping check, returned the expected known limitation). The database "
        "is ready to be queried in `03_sql_analysis.ipynb`.\n"
    )

    return "\n".join(lines)


report_markdown = build_database_validation_report(
    db_path=DB_PATH,
    tables=list_tables(conn),
    import_validation=import_validation,
    pk_results_df=pk_results_df,
    fk_results_df=fk_results_df,
    not_null_df=not_null_df,
    composite_key_violations=len(composite_key_check),
    review_cardinality_violations=len(review_cardinality_check),
    category_mapping_check=category_mapping_check,
    final_row_counts=final_row_counts,
)

report_path = REPORTS_DIR / "database_validation_report.md"
report_path.write_text(report_markdown, encoding="utf-8")
print(f"Validation report written to {report_path}")


Validation report written to /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics FINAL/reports/database_validation_report.md


## Section 9 — Performance Preparation (Recommendations Only — Not Created Yet)

No indexes are created at this point. The following are documented
recommendations, to be added once the actual query patterns from the SQL
analysis notebook are known to be slow enough to warrant them:

| Column(s) | Why | Expected benefit |
|---|---|---|
| `order_items.product_id` | Frequently joined to `products` for category/product-level revenue queries | Faster joins for category and product KPIs |
| `order_items.order_id` | Already part of the composite primary key, so it's implicitly indexed, but worth confirming query plans use it | Faster joins back to `orders` |
| `orders.customer_unique_id` | Frequently joined to `customers` for repeat-purchase and customer-revenue queries | Faster customer-level aggregations |
| `orders.order_purchase_timestamp` | Used in every monthly/time-series query in the SQL analysis notebook | Faster date-range filtering and `GROUP BY` on date parts |
| `products.category_id` | Frequently joined to `categories` for category-level revenue queries | Faster category rollups |
| `order_reviews.order_id` | Used to join review scores back to delivery performance | Faster review-to-order joins |

**Why these aren't created now:** premature indexing on a database this size
(tens of thousands of rows) adds write overhead and schema complexity with
no measurable read benefit yet. Indexes should only be added once actual
query plans (`EXPLAIN QUERY PLAN`) show they would help.


## Section 10 — Verification

Final confirmation that the database was built correctly.


In [18]:
print("Database builds successfully:", DB_PATH.exists())
print("Tables match the schema in sql/00_create_schema.sql:", set(list_tables(conn)) == set(TABLE_LOAD_ORDER))
print("Every cleaned CSV imported with matching row counts:", import_validation["match"].all())
print("All foreign keys resolve (except the documented informational category-mapping check):", fk_results_df["passed"].all())
print("No unexpected orphan records remain:", all_checks_passed)

conn.close()
print("\nConnection closed. Database build complete.")


Database builds successfully: True
Tables match the schema in sql/00_create_schema.sql: True
Every cleaned CSV imported with matching row counts: True
All foreign keys resolve (except the documented informational category-mapping check): True
No unexpected orphan records remain: True

Connection closed. Database build complete.


## Next Notebook

**`03_sql_analysis.ipynb`** will write and execute the full
~20-query SQL roadmap against this database: basic aggregations,
intermediate joins and grouping, and advanced junior-level queries using
CTEs, `ROW_NUMBER()`, `RANK()`/`DENSE_RANK()`, `SUM() OVER()` running
totals, and `LAG()` for month-over-month comparisons. Each query is saved
as a standalone `.sql` file in `sql/`, and its results are captured and
briefly interpreted — still without touching KPI dashboards, charts, or
business-report writing, which come later.
